In [ ]:
# import the libraries we need
import requests                 # talks to the API over HTTP
from pprint import pprint       # prints nested data in a readable way
import pandas as pd             # to work with DataFrames
import matplotlib.pyplot as plt # charts

# Base URL of the BibMedEd API running in Docker (docker compose up)
api_url = "http://localhost:8000/api"

# Create the bundled sample project on the server.
# POST / projects / sample creates it if missing , or returns the exisiting one.
# timeout = 30 stops the cell from hanging forever if the server is down.
project = requests.post(f"{api_url}/projects/sample", timeout = 30)

# stop with a clear error if the server returned 4xx or 5xx
project.raise_for_status()

# read the JSON body and take the project's id
# use this instead of hardcoding project_id =1, so it works on any fresh database.
project_id = project.json()["id"]

print(project_id)


In [ ]:
# Run the publications analysis. It must be POST, like authors and keywords.
publications_res = requests.post(f"{api_url}/projects/{project_id}/analysis/publications", timeout=30)
publications_res.raise_for_status()
publications_data = publications_res.json()
print(publications_data)

In [ ]:
# retriving year and its number of publishments 
year_count = pd.DataFrame(publications_data["results"]["yearly_counts"]).rename(columns = {"count":"NumberOfPublishedPaper"})

print(year_count)


In [ ]:
# plotting Bar chart
x = year_count["year"]
y = year_count["NumberOfPublishedPaper"]

plt.figure(figsize=(10,6))

bars = plt.bar(x,y, width = 0.65, color = "#4C78A8", edgecolor = "black", linewidth = 0.8)

plt.xlabel('Year', fontsize = 12)
plt.ylabel('Number of Published Papers', fontsize = 12)

plt.title("Published Papers By Years", fontsize = 16, fontweight = "bold")

plt.yticks(range(0, int(y.max()) + 1)) # logic : gives range from 0 upto max and increased by 1 because endpoint of range is exclusive
plt.ylim(0, y.max() * 1.1) # y.max()*1.1 because it adds 10% extra space above the tallest bar.

plt.grid(axis = "y", linestyle = "--", alpha = 0.3)

for bar in bars:
    height = bar.get_height()
    
    plt.text(
        bar.get_x()+ bar.get_width()/2,
        height+0.05,
        str(int(height)),
        ha = "center",
        fontsize = 11,
        fontweight = "bold")
    

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.figtext(
    0.5, 0.01, "Synthetic sample data",ha = "center", fontsize = 10, style = 'italic'
)

plt.tight_layout()
plt.show()

In [ ]:
# getting  authors 
authors_res = requests.post(f"{api_url}/projects/{project_id}/analysis/authors")
print(authors_res)

authors_data = authors_res.json()
pprint(authors_data)


In [ ]:
top_authors = pd.DataFrame(authors_data["results"]["top_authors"]).head(10)

pprint(top_authors)


In [ ]:
x = top_authors["name"]
y = top_authors["pub_count"]

plt.figure(figsize = (10,6))

bars = plt.barh(x,y,  color = "skyblue", edgecolor = "black", height=  0.8)

plt.xlabel("Total Count Of Publishment", fontsize = 12)
plt.ylabel("Authors", fontsize = 12 )

plt.grid(axis = "x", linestyle = "--", alpha = 0.3)

for bar in bars:
    plt.text(bar.get_width() + 0.1,              # x: just past the bar's end
             bar.get_y() + bar.get_height()/2,   # y: vertical middle of the bar
             str(int(bar.get_width())),          # the count, without decimals
             ha="left", va="center", fontsize=11, fontweight="bold")
    
plt.gca().spines["top"].set_visible(False)      # hide top border
plt.gca().spines["right"].set_visible(False)    # hide right border

plt.figtext(
    0.5, 0.01, "Synthetic sample data",ha = "center", fontsize = 10, style = 'italic'
)

plt.title("Top Authors by Publication Count", fontsize=16, fontweight="bold")
plt.tight_layout()    # fit labels and title inside the figure

plt.gca().invert_yaxis()
plt.show()


In [ ]:
keyword_res = requests.post(f"{api_url}/projects/{project_id}/analysis/keywords")
print(keyword_res)
keyword_data = keyword_res.json()
pprint(keyword_data)


In [ ]:
top_keywords = pd.DataFrame(keyword_data["results"]["top_keywords"]).rename(columns = {"term":"Keywords", "count" : "NumberOfTimesWordAppeared"}).sort_values("NumberOfTimesWordAppeared", ascending = False)
top_keywords

In [ ]:
y = top_keywords["NumberOfTimesWordAppeared"]
x =  top_keywords["Keywords"]

plt.figure(figsize = (10,6))

bars = plt.barh(x,y,  color = "skyblue", edgecolor = "black", height=  0.8)

plt.xlabel("Total Count Of Keyword Appearance", fontsize = 12)
plt.ylabel("Terms", fontsize = 12 )

plt.grid(axis = "x", linestyle = "--", alpha = 0.3)

for bar in bars:
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
             str(int(bar.get_width())), 
             ha="left", va="center", fontsize=11, fontweight="bold")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.figtext(
    0.5, 0.01, "Synthetic sample data",ha = "center", fontsize = 10, style = 'italic'
)

plt.title("Top Keywords by Appearance", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.gca().invert_yaxis()
plt.show()


In [ ]:
bundle_response = requests.get(f"{api_url}/projects/{project_id}/export/bundle", timeout=60)
bundle_response.raise_for_status()
with open("sample_project_export.zip", "wb") as f:
    f.write(bundle_response.content)
print(f"Downloaded zip, size: {len(bundle_response.content)} bytes")
